**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/pakhomov-2023-001025"], check=True)
    os.chdir("/content/encore/entries/pakhomov-2023-001025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Pakhomov et al. (2023) from dandiset 001025

**Paper.** Pakhomov, Gudvangen, Mangalanathan, Kondratiev, Redondo and Semenov (2023), "Next
generation CANCAN focusing for remote stimulation by nanosecond electric pulses",
Bioelectrochemistry 152, 108437, doi:10.1016/j.bioelechem.2023.108437.

**Dataset.** DANDI:001025, "The impact of voltage on CANCAN focusing of electroporation using
four-electrode arrays of varying sizes", version **0.240605.1826**
(doi:10.48324/dandi.001025/0.240605.1826). The paper cites no version of the dataset. This is the
only published version, and its 21 assets are the same as the draft's (same count and size), so it
is the most recent published version and the one pinned.

**Text read.** The claims file was written from the accepted author manuscript in PubMed Central
(PMC10247516), with no supplement and no linked code. This attempt read the same text.

**What the dataset holds.** One stitched YO-PRO-1 fluorescence image per dish, taken after the
nsEP exposure, with the first-pulse voltage and the number of packets per electrode in the
session description. There are no ROI measurements, electrode positions or pixel size, so the
notebook finds the electrodes in each image and places the paper's ROIs itself (helper
`cancan.py`). The images are the experiments of Fig. 7: Packet 2 rotated once (1Px4) at 2.4 to
5.6 kV, and rotated ten times (10Px4) at 4.8 and 6.4 kV.

| Claim | Tested here |
|---|---|
| fig4-unipolar-strongest | blocked, data_missing (single-cell time-lapse data not in the dandiset) |
| fig4-bipolar-cancellation | blocked, data_missing (as above) |
| fig5a-center-enhanced-without-rotation | blocked, other (no 40-packet or 40-train dishes here; the Fig. 5 images appear to be in dandiset 000686) |
| fig5bc-rotation-focuses-center | blocked, other (no 1Px40 or 1Tx40 dishes here; dandiset 000686) |
| fig6a-more-packets-more-uptake | blocked, other (no 5Px4 dishes; 1Px4 and 10Px4 are on different arrays here; dandiset 000875) |
| fig6b-normalized-center-by-packets | blocked, other (as above) |
| fig7a-amplitude-scales-uptake | tested |
| fig7b-targeting-independent-of-amplitude | tested |
| fig7cd-holds-for-large-array | tested (two of the paper's voltages, see below) |
| discussion-independent-of-array-size | tested (follows from the two above) |
| fig8a-packets-differ-near-electrodes | blocked, other (no Packet 1 or Packet 3 dishes here; dandiset 001027) |
| fig8b-packet2-best-targeting | blocked, other (as above) |
| fig9-gap-boosts-targeting | blocked, other (no dishes with the array above the monolayer here; dandiset 001028) |
| fig9c-restricted-to-center | blocked, other (as above) |

The images of Figs. 5, 6, 8 and 9 appear, from their titles and descriptions (printed below), to be
in four other dandisets that cite the paper (000686, 000875,
001027 and 001028); the claims file gives every claim as data from this dandiset, so those claims
are blocked here with `other`. The 4.0 kV 1Px4 and 6.4 kV 10Px4 dishes here match the Packet 2
conditions of Figs. 8 and 9 (for Fig. 8, if it used the small array, which its legend does not
say); their values are set against the paper's in a section after the
tested claims, without deciding those claims.

Random numbers are drawn only in the fig7b section, for a simulation of the chance of a false positive among its tests, from its own generator with a fixed seed (2023).

In [ ]:
import itertools
import time

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import remfile
from scipy import stats

import cancan as cc

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 40)
print("dandiset", cc.DANDISET, "version", cc.VERSION)

## Loading

Each image is streamed from the archive: the compressed HDF5 chunks of the YP channel (channel 0,
"FITC") are fetched with range requests and reduced, as they arrive, to sums and counts of non-zero
pixels in 4 x 4 blocks; nothing is written to disk. For each dish the electrodes are located on a
16 x 16 block image, and pixel sums are binned along the two diagonals of the array, from which
every ROI layout below is computed, and the radius of each needle footprint is measured. This cell takes about 12 minutes.

In [ ]:
assets = cc.list_assets()
res = {}
t0 = time.time()
for a in assets:
    url = cc.s3_url(a["asset_id"])
    f = h5py.File(remfile.File(url), "r")
    ims = f["acquisition/ScanningImaging"]
    info = dict(
        path=a["path"],
        subject=f["general/subject/subject_id"][()].decode(),
        start=f["session_start_time"][()].decode()[:19],
        description=f["session_description"][()].decode(),
        imaging=ims.attrs["description"],
        resolution_attr=float(ims["data"].attrs["resolution"]),
        unit=ims["data"].attrs["unit"],
        n_channels=ims["data"].shape[0],
    )
    f.close()
    info["kV"], info["reps"] = cc.parse_description(info["description"])
    s, n, st = cc.read_image(url, channel=0)
    img, zero = cc.coarse(s, n)
    size, pts, scores = cc.find_electrodes(img, zero)
    # strips wide and long enough for every ROI layout and pixel-size reading used below
    span = cc.SPAN_MM[size] * 1000 / cc.CAMERA_UM_PER_PX / cc.F
    fp = cc.footprint_radius(s, n, pts)  # dark footprint of each needle, to check the pixel size
    hists, bin_px = cc.strip_histograms(s, n, pts, 1.25 * span, 1.25 * span / (cc.N_ROI[size] / 2) + 2)
    res[a["path"]] = dict(info, size=size, pts=pts, scores=scores, hists=hists, fp=fp, img=img, zero=zero, stats=st,
                          nonzero_mean=s.sum() / max(n.sum(), 1))
    del s, n
print(f"{len(res)} images streamed and reduced in {(time.time() - t0) / 60:.1f} min")

## Data check

In [ ]:
rows = []
for p, r in res.items():
    d1 = np.hypot(*(r["pts"][0] - r["pts"][2])) * cc.FD
    d2 = np.hypot(*(r["pts"][1] - r["pts"][3])) * cc.FD
    rows.append(dict(
        file=p.split("/")[1].replace("_image.nwb", ""), subject=r["subject"], start=r["start"],
        kV=r["kV"], packets_per_electrode=r["reps"], channels=r["n_channels"],
        shape=r["stats"]["shape"], zero_frac=round(r["stats"]["zero_px"] / np.prod(r["stats"]["shape"][1:]), 3),
        max=r["stats"]["max"], mean_nonzero=round(r["nonzero_mean"], 1),
        array=r["size"], fit_small=round(r["scores"]["small"], 2), fit_large=round(r["scores"]["large"], 2),
        diag1_px=round(d1), diag2_px=round(d2),
    ))
dishes = pd.DataFrame(rows).sort_values(["packets_per_electrode", "kV", "start"]).reset_index(drop=True)
print(dishes.to_string())

print("\nDishes per condition (array found from the electrode spacing):")
print(dishes.groupby(["packets_per_electrode", "array", "kV"]).size().rename("dishes").to_string())
print("\nFiles per subject id:", dishes.subject.value_counts().to_dict())
print("Distinct subject ids:", dishes.subject.nunique(), "for", len(dishes), "files")
print("\nImaging descriptions:")
for k, v in pd.Series([r["imaging"].split("Additional metadata: ")[-1] for r in res.values()]).value_counts().items():
    print(f"  {v} files: {k}")
print("ImageSeries unit:", {r["unit"] for r in res.values()}, "; 'resolution' attribute:", {r["resolution_attr"] for r in res.values()})

# the array size is not in the metadata; check the geometric classification against the protocol
ok = ((dishes.packets_per_electrode == 1) & (dishes.array == "small")) | ((dishes.packets_per_electrode == 10) & (dishes.array == "large"))
print("\nEvery 1Px4 dish is on the small array and every 10Px4 dish on the large one (as Fig. 7 legend):", bool(ok.all()))

# pixel size: two readings
large = dishes[dishes.array == "large"]
small = dishes[dishes.array == "small"]
large_diag_px = large[["diag1_px", "diag2_px"]].values.mean()
small_diag_px = small[["diag1_px", "diag2_px"]].values.mean()
um = {"camera": cc.CAMERA_UM_PER_PX, "diag8.4": cc.LARGE_DIAGONAL_MM * 1000 / large_diag_px}
print(f"\nMean centre-to-centre diagonal: large {large_diag_px:.0f} px (sd {large[['diag1_px','diag2_px']].values.std():.0f}), "
      f"small {small_diag_px:.0f} px (sd {small[['diag1_px','diag2_px']].values.std():.0f}); ratio {large_diag_px / small_diag_px:.2f} "
      "(paper: 'approximately a 2-fold size difference')")
for k, v in um.items():
    print(f"  pixel size {k}: {v:.3f} um -> large diagonal {large_diag_px * v / 1000:.2f} mm, small {small_diag_px * v / 1000:.2f} mm, "
          f"image width {20480 * v / 1000:.1f} mm")
print("\nWhere the paper's ROI span ends against the needles (radius 0.75 mm, Section 2.3) and the paper's exclusion zone"
      " (5% of the diagonal, Section 2.4):")
for k, v in um.items():
    for size, dpx in (("small", small_diag_px), ("large", large_diag_px)):
        diag_mm = dpx * v / 1000
        edge = diag_mm / 2 - cc.NEEDLE_RADIUS_MM
        print(f"  {k}, {size} array: needle edge {edge:.2f} mm from the centre; span ends at {cc.SPAN_MM[size]} mm, "
              f"{edge - cc.SPAN_MM[size]:+.2f} mm from the needle edge (negative: on the needle); "
              f"exclusion zone {cc.EXCLUDE_FRAC * diag_mm:.2f} mm")

# needle footprints: radius of the dark disk around each electrode centre, against the 0.75 mm needle radius
fpr = pd.DataFrame([dict(file=p.split("/")[1].replace("_image.nwb", ""), array=r["size"], kV=r["kV"], electrode=k,
                         r_mm_camera=x["r_px"] * cc.F * cc.CAMERA_UM_PER_PX / 1000, inside=x["inside"], outside=x["outside"])
                    for p, r in res.items() for k, x in enumerate(r["fp"])])
print("\nNeedle footprint radius at the camera scale (mm), per dish (4 electrodes), with the median a.u. inside and around:")
print(fpr.groupby(["array", "kV", "file"]).agg(r_min=("r_mm_camera", "min"), r_median=("r_mm_camera", "median"),
                                              r_max=("r_mm_camera", "max"), inside=("inside", "median"),
                                              outside=("outside", "median")).round(3).to_string())
print("Not found:", int(fpr.r_mm_camera.isna().sum()), "of", len(fpr))
for size, g in fpr.groupby("array"):
    med = g.r_mm_camera.median()
    print(f"  {size} array: median radius {med:.3f} mm (IQR {g.r_mm_camera.quantile(0.25):.3f} to {g.r_mm_camera.quantile(0.75):.3f}), "
          f"so a 0.75 mm needle radius implies {cc.CAMERA_UM_PER_PX * cc.NEEDLE_RADIUS_MM / med:.3f} um per pixel "
          f"(camera reading 0.650, 8.4 mm reading {um['diag8.4']:.3f})")
# per-array scale implied by the footprints, used only as a variant below
um["footprint"] = {size: cc.CAMERA_UM_PER_PX * cc.NEEDLE_RADIUS_MM / g.r_mm_camera.median() for size, g in fpr.groupby("array")}
for size, dpx in (("small", small_diag_px), ("large", large_diag_px)):
    v = um["footprint"][size]
    diag_mm = dpx * v / 1000
    edge = diag_mm / 2 - cc.NEEDLE_RADIUS_MM
    print(f"  footprint scale {v:.3f} um, {size} array: diagonal {diag_mm:.2f} mm (paper, large array: 8.4 mm); needle edge {edge:.2f} mm "
          f"from the centre, span ends {edge - cc.SPAN_MM[size]:+.2f} mm from it; exclusion zone {cc.EXCLUDE_FRAC * diag_mm:.2f} mm")

# duplicates: correlation of the 16x16 block images over the area imaged in both
names = list(res)
cmax = []
for i, j in itertools.combinations(range(len(names)), 2):
    a, b = res[names[i]], res[names[j]]
    m = (a["zero"] < 0.01) & (b["zero"] < 0.01)
    cmax.append((np.corrcoef(a["img"][m], b["img"][m])[0, 1], names[i].split("/")[1], names[j].split("/")[1]))
cmax.sort(reverse=True)
print("\nHighest correlations between two dishes' images (1 would be a duplicate):")
for c in cmax[:3]:
    print(f"  {c[0]:.3f} {c[1]} {c[2]}")
print("Identical mean intensities:", dishes.mean_nonzero.duplicated().sum())

# dark floor by imaging session (0.1st percentile of imaged pixels, 16x16 blocks)
dishes["floor_p0.1"] = [round(float(np.percentile(res[p]["img"][res[p]["zero"] < 0.01], 0.1)), 1) for p in
                        [next(q for q in res if q.split("/")[1].replace("_image.nwb", "") == f) for f in dishes.file]]
dishes["date"] = dishes.start.str[:10]
print("\nDark floor of the images (0.1st percentile, a.u.) by imaging date and condition:")
print(dishes.groupby(["date", "packets_per_electrode", "kV"])["floor_p0.1"].agg(["count", "min", "max"]).to_string())
print("\nNo data stream has a clock: each file is one image at one time point, so there is no timing to check.")

The metadata against the data:

- The session descriptions give the first-pulse voltage and the number of packets per electrode,
  but not the array. The electrode spacing separates the images into two groups whose diagonals
  differ by the factor printed above, and the group with the wider spacing holds exactly the 10Px4
  dishes, as the Fig. 7 legend has it (1Px4 on the small array, 10Px4 on the large one).
- No pixel size is given; the ImageSeries `resolution` attribute is 96 or 1, which is not a pixel
  size, and 7 files name no detector, objective or channel. With the 0.65 um pixel that the named
  camera and objective give, the large-array diagonal is the value printed above, against the
  paper's 8.4 mm. The needles give a check: the dark footprint around each needle has the radius
  printed above, against the 0.75 mm of the 1.5 mm needles. The footprint edge is soft, so this
  check is approximate. It rules out the 8.4 mm scale, at which the needle edge would lie inside
  the paper's ROI span on both arrays (printed above), so that scale is not used as a reading. It
  does not confirm the camera scale either: the footprints imply about 0.68 um per pixel on the
  small array and 0.75 um on the large one (printed above), and at 0.75 um the large diagonal
  would be still further from the paper's 8.4 mm. The camera scale is kept for the main reading,
  since it rests on the hardware the files name and puts the large diagonal closer to 8.4 mm than
  the footprint scale does; the footprint scale is shown as a variant. At the camera scale the span
  ends within the paper's 5% exclusion zone; a variant below drops the end ROIs that lie in it.
- Subject ids are reused: two subject ids each hold two dishes of different days and protocols,
  so a subject id does not identify a dish ("Each dish was used in one experiment", Section 2.1).
- No two images are duplicates (the highest correlation between any two is printed above).
- The dark floor of the images is not a fixed camera offset: the 2.4 kV dishes have a floor of 33
  to 47 a.u. against 62 for the 4.8 kV dishes imaged on the same day, and the large-array session
  78 to 81. The paper does not say whether a background was removed (see fig7a).
- Fig. 7 states n = 3 for every voltage; the dandiset has 4 dishes at 2.4 kV, 3 at the other small
  array voltages, 2 at 4.8 kV and 3 at 6.4 kV on the large array.

### Other dandisets that cite the paper

The claims of Figs. 5, 6, 8 and 9 need dishes this dandiset does not hold. Four other dandisets
cite the paper; their titles and descriptions are read here from the archive (their files are not
opened), to say where those images appear to be.


In [ ]:
PAPER_DOI = "10.1016/j.bioelechem.2023.108437"
for d in ("000686", "000875", "001027", "001028"):
    info = cc.requests.get(f"{cc.API}/dandisets/{d}/", timeout=60).json()
    v = (info.get("most_recent_published_version") or info["draft_version"])["version"]
    meta = cc.requests.get(f"{cc.API}/dandisets/{d}/versions/{v}/", timeout=60).json()
    cites = any(PAPER_DOI in (r.get("identifier", "") + r.get("url", "")).lower() for r in meta.get("relatedResource", []))
    n = cc.requests.get(f"{cc.API}/dandisets/{d}/versions/{v}/assets/?page_size=1", timeout=60).json()["count"]
    print(f"{d} version {v}, {n} assets, cites the paper: {cites}\n  title: {meta['name']}\n  description: {meta.get('description', '')[:300]}\n")


### Electrode positions and ROI lines

The fitted electrode footprints (red circles) and the ROI span along each diagonal under the main
pixel-size reading (diagonal 1 red, diagonal 2 cyan), on the 16 x 16 block images. Brightness is
clipped at the 1st and 99th percentiles of each image.

In [ ]:
order = [next(q for q in res if q.split("/")[1].replace("_image.nwb", "") == f) for f in dishes.file]
fig, axs = plt.subplots(4, 6, figsize=(24, 16))
for ax, p in zip(axs.ravel(), order):
    r = res[p]
    v = r["img"][r["zero"] < 0.5]
    ax.imshow(r["img"], cmap="gray", vmin=np.percentile(v, 1), vmax=np.percentile(v, 99))
    pts = r["pts"]
    ctr = pts.mean(0)
    for q in pts:
        ax.add_patch(plt.Circle(q, cc.DISK_R[r["size"]], fill=False, color="r"))
    span = cc.SPAN_MM[r["size"]] * 1000 / um["camera"] / cc.FD
    for (i, j), col in (((0, 2), "r"), ((1, 3), "c")):
        e = (pts[j] - pts[i]) / np.hypot(*(pts[j] - pts[i]))
        ax.plot([ctr[0] - span * e[0], ctr[0] + span * e[0]], [ctr[1] - span * e[1], ctr[1] + span * e[1]], col, lw=1)
    m = np.ptp(pts, 0).max() * 0.75
    ax.set_xlim(ctr[0] - m, ctr[0] + m)
    ax.set_ylim(ctr[1] + m, ctr[1] - m)
    ax.set_title(f"{p.split('/')[1].replace('_image.nwb', '')}  {r['kV']} kV  {r['reps']}Px4", fontsize=10)
    ax.axis("off")
for ax in axs.ravel()[len(order):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Readings of the ROI layout

The claims file's opening comment names the readings the paper allows; one more is added here,
the pixel size, which the dataset does not give (helper docstring). Each combination is run:

- pixel size: `camera` (0.65 um, from the camera and objective the files name). The scale at which
  the large array's mean diagonal is 8.4 mm was a reading in round 1 and is dropped: it puts the
  end ROIs on the needles (data check). The scale implied by the needle footprints (about 0.68 um
  on the small array, 0.75 um on the large) is shown as a variant of the main reading;
- ROI count: `both` (92 or 224 ROIs over both diagonals, so 46 or 112 per diagonal) or `per` (92 or
  224 per diagonal);
- diagonals: `both` averaged, or `d1` or `d2` alone (the files do not say which electrode is e1);
- centre ROI: `a` one ROI centred on the array centre, `b` the mean of the two grid ROIs beside it;
- source: `raw` ROI means or a `loess` fit (span 0.75) along each diagonal (for LOESS the two centre
  readings coincide, so only `a` is listed);
- normalisation to the most peripheral ROI: `a` the mean of the end ROIs, `b` each half-diagonal
  to its own end ROI, then averaged.

This gives 18 layouts, and 36 readings with the two normalisations. The **main reading** is
camera, both, both, a, raw, a. Its reasons: the camera and objective are named in the files, and it
puts the large diagonal closer to the paper's 8.4 mm than the footprint scale does; the Fig. 5B legend averages both diagonals for rotated packets, and all Fig. 7
treatments are rotated; the Methods give one count for the measurement; the paper's numbers are
measurements, not fits; and "in percent to the most peripheral ROI" names one reference value. It is
an assumption wherever the readings disagree.

Three variants are shown beside the readings, none part of the criteria: the footprint
background subtracted from every ROI, the end ROIs that fall inside the paper's 5% exclusion
zone dropped (`cc.inner_values`, main reading otherwise), and the pixel size implied by the
needle footprints (main reading otherwise, `FP`).

In [ ]:
def measures(scale, count):
    out = {}
    for p, r in res.items():
        size = r["size"]
        u = um[scale][size] if isinstance(um[scale], dict) else um[scale]
        span = cc.SPAN_MM[size] * 1000 / u / cc.F
        n_diag = cc.N_ROI[size] // 2 if count == "both" else cc.N_ROI[size]
        out[p] = [cc.diagonal_measures(h, 1.0, span, n_diag) for h in r["hists"]]
    return out


M = {(s, c): measures(s, c) for s in cc.SCALES for c in cc.COUNTS}
READINGS = [r for r in itertools.product(cc.SCALES, cc.COUNTS, cc.DIAGONALS, cc.CENTERS, cc.SOURCES)
            if not (r[4] == "loess" and r[3] == "b")]
MAIN = ("camera", "both", "both", "a", "raw")
# variant: main reading at the per-array scale implied by the needle footprints (data check)
M[("footprint", "both")] = measures("footprint", "both")
FP = ("footprint", "both", "both", "a", "raw")
RCOLS = ["scale", "count", "diagonals", "centre", "source"]


def table(reps, reading, background=None):
    # per-dish values under one reading; background: dict path -> a.u. to subtract (variant only)
    scale, count, diag, center, source = reading
    rows = []
    for p, dm in M[(scale, count)].items():
        r = res[p]
        if r["reps"] != reps:
            continue
        if background is not None:
            b = background[p]
            dm = [{**d, source: {k: v - b for k, v in d[source].items()}} for d in dm]
        rows.append(dict(file=p.split("/")[1].replace("_image.nwb", ""), kV=r["kV"], **cc.summarize(dm, diag, center, source)))
    return pd.DataFrame(rows).sort_values(["kV", "file"]).reset_index(drop=True)


# background variant: median of the 16x16 blocks inside the electrode footprints (half radius),
# where the needles stood on the glass; a choice made here, used only as a variant
background = {}
for p, r in res.items():
    yy, xx = np.mgrid[: r["img"].shape[0], : r["img"].shape[1]]
    inside = np.zeros(r["img"].shape, bool)
    for q in r["pts"]:
        inside |= np.hypot(xx - q[0], yy - q[1]) < 0.5 * cc.DISK_R[r["size"]]
    background[p] = float(np.median(r["img"][inside & (r["zero"] < 0.01)]))
# exclusion-zone variant: per dish, values with the end ROIs inside the 5% zone dropped (main reading otherwise)
INNER, INNER_FP = {}, {}
for p, r in res.items():
    hd = [np.hypot(*(r["pts"][j] - r["pts"][i])) * cc.FD / cc.F / 2 for i, j in ((0, 2), (1, 3))]
    INNER[p] = cc.inner_values(r["hists"], M[MAIN[:2]][p], hd, um["camera"])
    INNER_FP[p] = cc.inner_values(r["hists"], M[FP[:2]][p], hd, um["footprint"][r["size"]])
print("Exclusion-zone variant, ROIs dropped at each end (both diagonals) by array:",
      pd.DataFrame([dict(array=res[p]["size"], dropped=str(v["dropped"])) for p, v in INNER.items()])
      .groupby("array").dropped.agg(lambda s: s.value_counts().to_dict()).to_dict())
print("At the footprint scale:",
      pd.DataFrame([dict(array=res[p]["size"], dropped=str(v["dropped"])) for p, v in INNER_FP.items()])
      .groupby("array").dropped.agg(lambda s: s.value_counts().to_dict()).to_dict())


def inner_table(reps, background=None, inner=None):
    rows = []
    for p, v in (INNER if inner is None else inner).items():
        r = res[p]
        if r["reps"] != reps:
            continue
        b = 0.0 if background is None else background[p]
        c, h, e = v["center"] - b, v["half"] - b, v["periph"] - b
        rows.append(dict(file=p.split("/")[1].replace("_image.nwb", ""), kV=r["kV"], periph=e, half=h, center=c,
                         nc_a=100 * c / e, nh_a=100 * h / e))
    return pd.DataFrame(rows).sort_values(["kV", "file"]).reset_index(drop=True)


print("Footprint background (a.u.) by condition:")
print(pd.DataFrame([dict(reps=r["reps"], kV=r["kV"], bg=background[p]) for p, r in res.items()])
      .groupby(["reps", "kV"]).bg.agg(["count", "mean", "min", "max"]).round(1).to_string())

### Profiles

Mean normalised profiles (main reading, both diagonals averaged and folded about the centre so
that 0 is the centre and 1 the most peripheral ROI), mean +/- s.e. over dishes, for each voltage.

In [ ]:
def folded_profiles(reps, scale="camera", count="both"):
    # per dish: mean over the 4 half-diagonals of the raw ROI profile, from centre to periphery
    out = {}
    for p, dm in M[(scale, count)].items():
        r = res[p]
        if r["reps"] != reps:
            continue
        halves = []
        for d in dm:
            prof = d["raw"]["profile"]
            h = len(prof) // 2
            halves += [prof[:h][::-1], prof[h:]]
        out[p] = np.nanmean(halves, axis=0)
    return out


fig, axs = plt.subplots(2, 2, figsize=(13, 9))
for col, reps in enumerate((1, 10)):
    prof = folded_profiles(reps)
    for kv in sorted({res[p]["kV"] for p in prof}):
        ps = [p for p in prof if res[p]["kV"] == kv]
        P = np.array([prof[p] for p in ps])
        x = np.linspace(0, 1, P.shape[1])
        Pn = 100 * P / P[:, -1:]
        for row, Q in enumerate((P, Pn)):
            m, se = Q.mean(0), Q.std(0, ddof=1) / np.sqrt(len(Q))
            axs[row, col].errorbar(x, m, se, label=f"{kv} kV (n={len(ps)})", lw=1, capsize=0)
            print(f"{reps}Px4 {kv} kV {'a.u.' if row == 0 else '% periphery'}: range of means {m.min():.1f} to {m.max():.1f}")
    axs[0, col].set_title(f"{'small' if reps == 1 else 'large'} array, {reps}Px4, a.u.")
    axs[1, col].set_title(f"{'small' if reps == 1 else 'large'} array, {reps}Px4, % of most peripheral ROI")
    for row in (0, 1):
        axs[row, col].set_xlabel("distance from centre, fraction of ROI span")
        axs[row, col].legend(fontsize=8)
plt.tight_layout()
plt.show()

## fig7a-amplitude-scales-uptake

> On the small array with 1Px4, raising the first pulse amplitude from 2.4 to 5.6 kV increases YP
> fluorescence 15.5 times at the most peripheral point, 16.2 times half-way to the center and 16.8
> times in the center.

Criterion: ratios of the 5.6 kV mean to the 2.4 kV mean over dishes, in a.u., with no background
subtracted; each within 10% (15.5 +/- 1.55, 16.2 +/- 1.62, 16.8 +/- 1.68) under a reading of the
opening comment; partially reproduced when one or two are.

In [ ]:
TOL = {"periph": (15.5, 1.55), "half": (16.2, 1.62), "center": (16.8, 1.68)}


def ratios(reading, background=None, drop=None):
    df = table(1, reading, background)
    if drop is not None:
        df = df[df.file != drop]
    g = df.groupby("kV")[["periph", "half", "center"]].mean()
    return g.loc[5.6] / g.loc[2.4], df


rows = []
for rd in READINGS:
    rat, _ = ratios(rd)
    rows.append(dict(zip(RCOLS, rd), **{f"ratio_{k}": round(rat[k], 2) for k in TOL},
                     n_within=int(sum(abs(rat[k] - v) <= t for k, (v, t) in TOL.items()))))
r7a = pd.DataFrame(rows)
print(r7a.to_string())
print("\nReadings with at least one ratio within tolerance:", int((r7a.n_within > 0).sum()), "of", len(r7a))
print("Range over readings: periphery", r7a.ratio_periph.min(), "to", r7a.ratio_periph.max(),
      "; half-way", r7a.ratio_half.min(), "to", r7a.ratio_half.max(),
      "; centre", r7a.ratio_center.min(), "to", r7a.ratio_center.max())

In [ ]:
rat, df = ratios(MAIN)
print("Main reading, per dish (a.u.):")
print(df[df.kV.isin([2.4, 5.6])][["file", "kV", "periph", "half", "center"]].round(1).to_string())
g = df.groupby("kV")[["periph", "half", "center"]].agg(["mean", "sem", "count"]).round(1)
print("\nMean, s.e. and n by voltage (all five voltages):")
print(g.to_string())
print("\nMain reading ratios 5.6/2.4 kV:", rat.round(2).to_dict())

print("\n2.4 kV has 4 dishes where the paper states n = 3; ratios with each one left out:")
for f in df[df.kV == 2.4].file:
    r_, _ = ratios(MAIN, drop=f)
    print(f"  without {f}: {r_.round(2).to_dict()}")

print("\nVariant, footprint background subtracted from every ROI (not the criterion):")
rows = []
for rd in READINGS:
    r_, _ = ratios(rd, background)
    rows.append(dict(zip(RCOLS, rd), **{f"ratio_{k}": round(r_[k], 2) for k in TOL},
                     n_within=int(sum(abs(r_[k] - v) <= t for k, (v, t) in TOL.items()))))
r7a_bg = pd.DataFrame(rows)
print(r7a_bg[r7a_bg[RCOLS].apply(tuple, axis=1) == MAIN].to_string())
print("Readings with all three within tolerance:", int((r7a_bg.n_within == 3).sum()), "of", len(r7a_bg),
      "; with at least one:", int((r7a_bg.n_within > 0).sum()))
print("Range over readings: periphery", r7a_bg.ratio_periph.min(), "to", r7a_bg.ratio_periph.max(),
      "; half-way", r7a_bg.ratio_half.min(), "to", r7a_bg.ratio_half.max(),
      "; centre", r7a_bg.ratio_center.min(), "to", r7a_bg.ratio_center.max())
print("\nVariant, end ROIs inside the 5% exclusion zone dropped (main reading otherwise):")
for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    d = inner_table(1, bg)
    gi = d.groupby("kV")[["periph", "half", "center"]].mean()
    ri = gi.loc[5.6] / gi.loc[2.4]
    print(f"  {bgname}: ratios {ri.round(2).to_dict()}, within tolerance: "
          f"{int(sum(abs(ri[k] - v) <= t for k, (v, t) in TOL.items()))} of 3")

print("\nVariant, pixel size from the needle footprints (main reading otherwise; not the criterion):")
for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    r_, _ = ratios(FP, bg)
    d = inner_table(1, bg, INNER_FP)
    gi = d.groupby("kV")[["periph", "half", "center"]].mean()
    ri = gi.loc[5.6] / gi.loc[2.4]
    print(f"  {bgname}: ratios {r_.round(2).to_dict()}, within tolerance {int(sum(abs(r_[k] - v) <= t for k, (v, t) in TOL.items()))} of 3; "
          f"with the exclusion-zone ROIs dropped {ri.round(2).to_dict()}, within tolerance {int(sum(abs(ri[k] - v) <= t for k, (v, t) in TOL.items()))} of 3")


**Result: not reproduced.** Without background subtraction, as the criterion fixes, the 5.6 kV over
2.4 kV ratios under the main reading are 5.99 at the most peripheral ROI, 5.65 half-way and 7.25 at
the centre, against 15.5, 16.2 and 16.8, and none of the 18 readings brings any ratio within
tolerance (the largest is 7.59). Leaving out any one of the four 2.4 kV dishes changes the ratios by
at most 0.59, and dropping the end ROIs that lie in the paper's exclusion zone gives 5.77, 5.84 and
7.25. Fluorescence rises with voltage at every position. With the median inside the needle
footprints subtracted as a background (150.7 a.u. on average at 2.4 kV), a variant the criterion
excludes, the main reading gives 13.37, 15.87 and 15.16, two of three within tolerance, 2 of 18
readings put all three within it, and with the exclusion-zone ROIs also dropped all three are within
it (14.88, 17.10 and 15.16). At the pixel size implied by the needle footprints the ratios are
5.71, 5.45 and 7.32 without background (none within tolerance) and 14.17, 14.82 and 15.39 with it
(all three within). The outcome turns on whether an offset was removed. The claims file
says none was, as in the Methods, but the Methods do not say so; this is asked of a reader in
`questions/001025.yaml`.

## fig7b-targeting-independent-of-amplitude

> Normalized to the most peripheral ROI, the profiles for different first pulse amplitudes fully
> overlap, so the spatial distribution of electroporation, and NG-CANCAN targeting, does not depend
> on amplitude within the studied range.

Criterion: a claim of absence. Small array, 1Px4, every voltage (2.4, 3.2, 4.0, 4.8 and 5.6 kV, from
the dataset). The normalised values at the centre ROI and at the half-way ROI are compared between
every pair of voltages by the two-sided Student's t-test, no correction. Reproduced when no pair
differs at p < 0.05; not reproduced when any does.

In [ ]:
def pair_tests(reps, reading, norm="a", background=None):
    df = table(reps, reading, background)
    kvs = sorted(df.kV.unique())
    out = []
    for pos, lab in (("nc", "centre"), ("nh", "half-way")):
        col = f"{pos}_{norm}"
        for a, b in itertools.combinations(kvs, 2):
            x, y = df[df.kV == a][col], df[df.kV == b][col]
            t, p = stats.ttest_ind(x, y)
            out.append(dict(position=lab, kV_a=a, kV_b=b, n_a=len(x), n_b=len(y), mean_a=x.mean(), mean_b=y.mean(),
                            diff=y.mean() - x.mean(), t=t, p=p))
    return pd.DataFrame(out)


def readings_summary(reps, background=None):
    rows = []
    for rd in READINGS:
        for norm in "ab":
            t = pair_tests(reps, rd, norm, background)
            rows.append(dict(zip(RCOLS, rd), norm=norm, n_tests=len(t), n_p_below_05=int((t.p < 0.05).sum()),
                             min_p=t.p.min(), max_abs_diff=t["diff"].abs().max()))
    return pd.DataFrame(rows)


t7b = pair_tests(1, MAIN, "a")
print("Main reading (normalisation a), every pair of voltages, % of the most peripheral ROI:")
print(t7b.round(4).to_string())
print(f"\nPairs with p < 0.05: {(t7b.p < 0.05).sum()} of {len(t7b)}")
df = table(1, MAIN)
print("\nPer dish, normalised centre and half-way (%):")
print(df[["file", "kV", "nc_a", "nh_a", "nc_b", "nh_b"]].round(1).to_string())

print("\n2.4 kV has 4 dishes where the paper states n = 3; each left out (main reading, normalisation a):")
for f in df[df.kV == 2.4].file:
    d = df[df.file != f]
    sig = []
    for pos, lab in (("nc_a", "centre"), ("nh_a", "half-way")):
        for a, b in itertools.combinations(sorted(d.kV.unique()), 2):
            _, p = stats.ttest_ind(d[d.kV == a][pos], d[d.kV == b][pos])
            if p < 0.05:
                sig.append(f"{lab} {a} vs {b} kV p = {p:.4f}")
    print(f"  without {f}: {len(sig)} pairs at p < 0.05: {sig}")

In [ ]:
s7b = readings_summary(1)
print(f"Every reading ({len(s7b)} = {len(READINGS)} layouts x 2 normalisations):")
print(s7b.round(4).to_string())
print("\nReadings with no pair at p < 0.05:", int((s7b.n_p_below_05 == 0).sum()), "of", len(s7b))
print("Number of significant pairs over readings:", s7b.n_p_below_05.value_counts().sort_index().to_dict())

# chance of at least one p < 0.05 among the 20 tests when no effect exists. The 10 pairs share groups and
# the two positions are measured on the same dishes, so the tests are not independent: simulate the design
# (group sizes of the dataset, normal values) with the two positions independent, identical, and correlated
# as in the data (within-voltage correlation of the normalised centre and half-way values, main reading)
rng = np.random.default_rng(2023)
d0 = table(1, MAIN)
ns = d0.groupby("kV").size().to_dict()
resid = d0[["nc_a", "nh_a"]] - d0.groupby("kV")[["nc_a", "nh_a"]].transform("mean")
rho = float(np.corrcoef(resid.nc_a, resid.nh_a)[0, 1])
NSIM = 20000


def fwer(rho_):
    z = {kv: rng.multivariate_normal([0, 0], [[1, rho_], [rho_, 1]], size=(NSIM, n)) for kv, n in ns.items()}
    anyp = np.zeros(NSIM, bool)
    for a, b in itertools.combinations(sorted(ns), 2):
        for k in (0, 1):
            anyp |= stats.ttest_ind(z[a][:, :, k], z[b][:, :, k], axis=1).pvalue < 0.05
    return anyp.mean()


print(f"Chance of at least one p < 0.05 among the 20 tests with no effect (group sizes {ns}, {NSIM} simulations, seed 2023):")
print(f"  20 independent tests (formula): {1 - 0.95 ** 20:.2f}")
for lab, r_ in (("positions independent", 0.0), (f"positions correlated as in the data (r = {rho:.2f})", rho), ("positions identical (10 tests)", 1.0)):
    print(f"  simulated, {lab}: {fwer(min(r_, 0.999999)):.2f}")
print(f"Bonferroni threshold for 20 tests: {0.05 / 20:.4f} (the criterion applies no correction); smallest p, main reading: {t7b.p.min():.4f}")

# variant: test at every ROI position of the folded normalised profile
prof = folded_profiles(1)
kvs = sorted({res[p]["kV"] for p in prof})
P = {kv: np.array([100 * prof[p] / prof[p][-1] for p in prof if res[p]["kV"] == kv]) for kv in kvs}
npos = P[kvs[0]].shape[1]
sig = np.zeros(npos, int)
for a, b in itertools.combinations(kvs, 2):
    _, p = stats.ttest_ind(P[a][:, :-1], P[b][:, :-1], axis=0)
    sig[:-1] += p < 0.05
print(f"\nVariant over all ROIs (folded profile, {npos - 1} positions x 10 pairs = {(npos - 1) * 10} tests; the last position is the 100% reference): "
      f"{sig.sum()} at p < 0.05; positions with any: {int((sig > 0).sum())}")

s7b_bg = readings_summary(1, background)
print("\nVariant with the footprint background subtracted, every reading:")
print(s7b_bg.round(4).to_string())
m = s7b_bg[(s7b_bg[RCOLS].apply(tuple, axis=1) == MAIN) & (s7b_bg.norm == "a")]
print("Main reading:", m[["n_p_below_05", "min_p"]].to_dict("records"),
      "; readings with no pair at p < 0.05:", int((s7b_bg.n_p_below_05 == 0).sum()), "of", len(s7b_bg))
print("\nMain reading with background subtracted, every pair:")
print(pair_tests(1, MAIN, "a", background).round(4).to_string())
print("\nPer dish, normalised centre and half-way with background subtracted (%):")
print(table(1, MAIN, background)[["file", "kV", "nc_a", "nh_a", "nc_b", "nh_b"]].round(1).to_string())


def inner_pairs(reps, bg=None, inner=None):
    d = inner_table(reps, bg, inner)
    out = []
    for pos, lab in (("nc_a", "centre"), ("nh_a", "half-way")):
        for a, b in itertools.combinations(sorted(d.kV.unique()), 2):
            x, y = d[d.kV == a][pos], d[d.kV == b][pos]
            t, p = stats.ttest_ind(x, y)
            out.append(dict(position=lab, kV_a=a, kV_b=b, mean_a=x.mean(), mean_b=y.mean(), p=p))
    return pd.DataFrame(out)


for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    t = pair_tests(1, FP, "a", bg)
    print(f"\nVariant, pixel size from the needle footprints, {bgname}: {(t.p < 0.05).sum()} of {len(t)} pairs at p < 0.05; "
          f"smallest p {t.p.min():.4f}")
    print(t[t.p < 0.05].round(4).to_string())
    t = inner_pairs(1, bg, INNER_FP)
    print(f"  and with the exclusion-zone ROIs dropped: {(t.p < 0.05).sum()} of {len(t)} pairs at p < 0.05; smallest p {t.p.min():.4f}")

for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    t = inner_pairs(1, bg)
    print(f"\nVariant, end ROIs inside the exclusion zone dropped, {bgname}: {(t.p < 0.05).sum()} of {len(t)} pairs at p < 0.05; "
          f"smallest p {t.p.min():.4f}; normalised centre means by kV "
          f"{inner_table(1, bg).groupby('kV').nc_a.mean().round(1).to_dict()}")
    print(t[t.p < 0.05].round(4).to_string())

**Result: not reproduced, resting on the main reading.** Under the main reading 1 of 20 pairs
differs at p < 0.05: the normalised centre, 108.6% at 2.4 kV against 136.9% at 4.8 kV (p = 0.0037).
The half-way values do not differ (smallest p = 0.12). The pair stays significant whichever 2.4 kV
dish is left out (p = 0.0012 to 0.0116). The readings disagree: 10 of 36 give no significant pair
and 26 give one to five. With no effect, the chance of at least one p below 0.05 among the 20
uncorrected tests is 0.64 if the tests were independent; they share groups and dishes, and the
simulation of this design gives 0.47 with the two positions correlated as in the data (0.50 if
they were independent). So a single significant pair is common by chance, though less common than
0.64 suggests. The p = 0.0037 would not pass a Bonferroni threshold for 20 tests (0.0025), which
the criterion does not apply. The normalised centre means lie between 108.6% and 136.9% across
voltages. The result is fragile in
the other direction too: with the footprint background subtracted (a variant the claims file
excludes), the main reading finds no pair (smallest p = 0.18) and 30 of 36 readings find none.
With the end ROIs inside the exclusion zone dropped, 4 of 20 pairs differ without background and 2
of 20 with it. At the pixel size implied by the needle footprints, 1 of 20 pairs differs without
background (the same pair, p = 0.0279) and none with it (smallest p = 0.14).

## fig7cd-holds-for-large-array

> The independence of the normalized distribution from amplitude also holds when rotating multiple
> packets (10Px4) on the larger array.

Criterion: as fig7b, on the large array with 10Px4 (n = 3), with the same test, positions and rule;
the rise of a.u. values with voltage at the three positions is reported. The dataset holds two
voltages for this array, 4.8 kV (2 dishes) and 6.4 kV (3 dishes); the paper's other voltages for
Fig. 7C and 7D are not in it, so one pair is tested.

In [ ]:
t7cd = pair_tests(10, MAIN, "a")
print("Main reading (normalisation a):")
print(t7cd.round(4).to_string())
df = table(10, MAIN)
print("\nPer dish:")
print(df[["file", "kV", "periph", "half", "center", "nc_a", "nh_a", "nc_b", "nh_b"]].round(1).to_string())
g = df.groupby("kV")[["periph", "half", "center"]].mean()
print("\nRise in a.u. from 4.8 to 6.4 kV (ratio of means):", (g.loc[6.4] / g.loc[4.8]).round(2).to_dict())
s7cd = readings_summary(10)
print("\nEvery reading:")
print(s7cd.round(4).to_string())
print("\nReadings with no pair at p < 0.05:", int((s7cd.n_p_below_05 == 0).sum()), "of", len(s7cd))
s7cd_bg = readings_summary(10, background)
print("\nVariant with the footprint background subtracted, every reading:")
print(s7cd_bg.round(4).to_string())
print("Readings with no pair at p < 0.05:", int((s7cd_bg.n_p_below_05 == 0).sum()), "of", len(s7cd_bg))
print("\nMain reading with background subtracted:")
print(pair_tests(10, MAIN, "a", background).round(4).to_string())
print("Per dish, with background subtracted:")
print(table(10, MAIN, background)[["file", "kV", "periph", "half", "center", "nc_a", "nh_a", "nc_b", "nh_b"]].round(1).to_string())
for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    t = inner_pairs(10, bg)
    print(f"\nVariant, end ROIs inside the exclusion zone dropped, {bgname}:")
    print(t.round(4).to_string())
for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    t = pair_tests(10, FP, "a", bg)
    ti = inner_pairs(10, bg, INNER_FP)
    print(f"\nVariant, pixel size from the needle footprints, {bgname}: centre p = {t.p.iloc[0]:.4f} "
          f"({t.mean_a.iloc[0]:.1f}% against {t.mean_b.iloc[0]:.1f}%), half-way p = {t.p.iloc[1]:.4f}; "
          f"with the exclusion-zone ROIs dropped, centre p = {ti.p.iloc[0]:.4f}, half-way p = {ti.p.iloc[1]:.4f}")


**Result: not reproduced, resting on the main reading.** Under the main reading the normalised
centre is 232.6% at 4.8 kV (2 dishes) and 126.0% at 6.4 kV (3 dishes), p = 0.0494; the half-way
values do not differ (p = 0.16). Both 4.8 kV dishes (270.6% and 194.7%) lie above all three 6.4 kV
dishes (102.4% to 149.8%). By the criterion the absence is not shown. 27 of 36 readings find no
difference, and with the background subtracted the main reading gives p = 0.0705 (25 of 36 readings
find none), so the outcome rests on the main reading and on a two-dish group; with two dishes
against three a non-significant reading could not show absence either. With the end ROIs inside
the exclusion zone dropped the centre differs more clearly (p = 0.0173, and 0.0089 with background
subtracted). At the pixel size implied by the needle footprints the difference is gone: 249.5%
against 226.6%, p = 0.57 (p = 0.25 with background subtracted), so the outcome rests on the camera
scale as well. The a.u. values rise from 4.8 to 6.4 kV at all three positions (ratios 3.53 at the
periphery, 1.89 half-way and 2.00 at the centre).

## discussion-independent-of-array-size

> Remote targeting, within the studied limits, depends neither on pulse amplitude nor on array
> size, so findings from small arrays can be extrapolated to larger inter-electrode distances.

Criterion: reproduced when fig7b-targeting-independent-of-amplitude and fig7cd-holds-for-large-array
are both reproduced. The normalised profiles of the two arrays are shown side by side, on distance
as a fraction of the ROI span; the small array received 1Px4 and the large array 10Px4, so the
comparison mixes array size with packet number.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for reps, ls in ((1, "-"), (10, "--")):
    prof = folded_profiles(reps)
    for kv in sorted({res[p]["kV"] for p in prof}):
        Q = np.array([100 * prof[p] / prof[p][-1] for p in prof if res[p]["kV"] == kv])
        x = np.linspace(0, 1, Q.shape[1])
        ax.plot(x, Q.mean(0), ls, label=f"{'small' if reps == 1 else 'large'} {reps}Px4 {kv} kV")
        print(f"{'small' if reps == 1 else 'large'} {kv} kV: centre {Q.mean(0)[0]:.1f}%, minimum {Q.mean(0).min():.1f}% at "
              f"{x[np.argmin(Q.mean(0))]:.2f} of the span")
ax.set_xlabel("distance from centre, fraction of ROI span")
ax.set_ylabel("% of most peripheral ROI")
ax.legend(fontsize=7)
plt.show()
print("fig7b pairs at p < 0.05 (main):", int((t7b.p < 0.05).sum()), "; fig7cd pairs at p < 0.05 (main):", int((t7cd.p < 0.05).sum()))

**Result: not reproduced, resting on the main reading.** Both claims it depends on are not
reproduced under the main reading, each by one comparison near or below p = 0.05, so by its
criterion this claim is not reproduced. The side-by-side profiles cannot isolate array size,
since the arrays received different packet numbers.

## Packet 2 dishes against Figs. 8 and 9 (not a test of those claims)

The fig8 and fig9 claims are blocked here, because their criteria need Packets 1 and 3 and the
dishes with gaps, which are in other dandisets. Two conditions of this dandiset match conditions of
those figures: 4.0 kV 1Px4 is the Packet 2 group of Fig. 8 (if Fig. 8 used the small array, which
its legend does not say), and 6.4 kV 10Px4 on the large array is the 0 mm group of Fig. 9. Their
values are set against the paper's numbers for Packet 2 in Section 3.4 (about 120 a.u. near the
electrodes, 215 to 235 a.u. in the centre, 183% normalised centre), as a check of whether the
dataset's a.u. are on the paper's scale. The text gives no a.u. for the 0 mm group of Fig. 9, only
ratios to the gap groups, so that profile is printed without a reference.

In [ ]:
PAPER8 = dict(periph=120, center=(215, 235), nc=183)
for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    d = table(1, MAIN, bg)
    d = d[d.kV == 4.0]
    print(f"4.0 kV 1Px4, main reading, {bgname}:")
    print(d[["file", "periph", "half", "center", "nc_a", "nc_b"]].round(1).to_string())
    print(f"  mean periphery {d.periph.mean():.1f} a.u. (paper about {PAPER8['periph']}), centre {d.center.mean():.1f} a.u. "
          f"(paper {PAPER8['center'][0]} to {PAPER8['center'][1]}), normalised centre {d.nc_a.mean():.1f}% (a) / "
          f"{d.nc_b.mean():.1f}% (b), ratio of means {100 * d.center.mean() / d.periph.mean():.1f}% (paper {PAPER8['nc']}%)")
    di = inner_table(1, bg)
    di = di[di.kV == 4.0]
    print(f"  exclusion-zone variant: periphery {di.periph.mean():.1f}, centre {di.center.mean():.1f}, normalised centre {di.nc_a.mean():.1f}%")

for bgname, bg in (("no background", None), ("footprint background subtracted", background)):
    d = table(10, MAIN, bg)
    d = d[d.kV == 6.4]
    print(f"\n6.4 kV 10Px4 (Fig. 9, 0 mm), main reading, {bgname}: periphery {d.periph.mean():.1f}, half-way {d.half.mean():.1f}, "
          f"centre {d.center.mean():.1f} a.u.; normalised centre {d.nc_a.mean():.1f}% (per dish {d.nc_a.round(1).tolist()})")
prof = folded_profiles(10)
Q = np.array([prof[p] for p in prof if res[p]["kV"] == 6.4])
x = np.linspace(0, 1, Q.shape[1])
dbb = table(10, ("camera", "both", "both", "b", "raw"))
print("\n6.4 kV 10Px4, two definitions of the centre, % of the mean of the end ROIs, mean over dishes:\n"
      f"  centre reading a (one ROI centred on the array centre; the main reading): {table(10, MAIN).query('kV == 6.4').nc_a.mean():.1f}%\n"
      f"  centre reading b (mean of the two grid ROIs beside the centre, the first position of the folded profile): "
      f"{dbb[dbb.kV == 6.4].nc_a.mean():.1f}%")
Pn = 100 * Q / Q[:, -1:]
print(f"Folded normalised profile (each dish to its own end ROIs, as in the discussion section): first position {Pn.mean(0)[0]:.1f}%, "
      f"minimum {Pn.mean(0).min():.1f}% at {x[np.argmin(Pn.mean(0))]:.2f} of the span")
print("\n6.4 kV 10Px4 folded profile (no background), mean over 3 dishes at tenths of the span (0 centre, 1 periphery):")
print(pd.DataFrame(dict(position=x, au=Q.mean(0), pct=100 * Q.mean(0) / Q.mean(0)[-1])).iloc[np.searchsorted(x, np.linspace(0, 1, 11)).clip(0, len(x) - 1)]
      .round(2).to_string(index=False))

The 4.0 kV 1Px4 dishes are not on the paper's Fig. 8 scale: without background their periphery is
614.9 a.u. and their centre 718.8 a.u., against about 120 and 215 to 235 a.u., and their normalised
centre is 118.8% (ratio of means 116.9%) against 183%. With the footprint background subtracted
they are 353.8 and 457.7 a.u. and 137.3%, still far from the paper's values. Either Fig. 8's Packet 2
dishes are not these dishes, or the paper's a.u. were scaled or corrected in a way the text does not
describe. For the 6.4 kV 10Px4 dishes the centre ROI of the main reading is 126.0% of the end ROIs; the
first position of the folded profile, which is the mean of the two grid ROIs beside the centre
(centre reading b) and is the value given in the discussion section, is 145.0%. The folded profile
has its minimum, 40.9%, at 0.76 of the span. The text gives no a.u. for this group to compare
with. These numbers decide none of the claims.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig4-unipolar-strongest | blocked (data_missing) |
| fig4-bipolar-cancellation | blocked (data_missing) |
| fig5a-center-enhanced-without-rotation | blocked (other: data in dandiset 000686) |
| fig5bc-rotation-focuses-center | blocked (other: data in dandiset 000686) |
| fig6a-more-packets-more-uptake | blocked (other: data in dandiset 000875) |
| fig6b-normalized-center-by-packets | blocked (other: data in dandiset 000875) |
| fig7a-amplitude-scales-uptake | not reproduced |
| fig7b-targeting-independent-of-amplitude | not reproduced (assumed: main reading) |
| fig7cd-holds-for-large-array | not reproduced (assumed: main reading) |
| discussion-independent-of-array-size | not reproduced (assumed: main reading) |
| fig8a-packets-differ-near-electrodes | blocked (other: data in dandiset 001027) |
| fig8b-packet2-best-targeting | blocked (other: data in dandiset 001027) |
| fig9-gap-boosts-targeting | blocked (other: data in dandiset 001028) |
| fig9c-restricted-to-center | blocked (other: data in dandiset 001028) |

How the analysis differs from the paper's:

- The paper measured ROIs in MetaMorph on images whose electrode positions the experimenters knew;
  here the electrodes are found by fitting four dark disks, and the ROIs placed from them.
- The pixel size is not in the dataset; the camera scale (0.65 um) is used. The needle footprints
  rule out the 8.4 mm scale of round 1, which would put the end ROIs on the needles, but do not
  confirm the camera scale: they imply about 0.68 um (small array) and 0.75 um (large array). At
  that scale, a variant, fig7cd's centre difference disappears (p = 0.57).
- At the camera scale the paper's ROI spans end inside its own 5% exclusion zone; the spans are kept
  as stated, and a variant drops the end ROIs inside the zone.
- ROI width across the diagonal is not given; square ROIs are used.
- Whether a background was subtracted is not stated; the claims file fixes none, and the footprint
  background is shown only as a variant. It changes fig7a from about 6-fold to near the paper's
  16-fold and removes the one significant pair of fig7b. A reader is asked whether the claims file
  reads the Methods correctly here.
- 2.4 kV has four dishes here, against n = 3 in the paper; all four are used, with leave-one-out
  shown for fig7a and fig7b.
- The large array has two voltages here (4.8 kV with 2 dishes and 6.4 kV with 3).
- The packet is not named in the files; Packet 2 is assumed, as in Fig. 7 and the dandiset's title.

What was found about the dataset:

- It holds only the images of Fig. 7: no single-cell time-lapse data (Fig. 4) and none of the
  experiments of Figs. 5, 6, 8 and 9 (other than, possibly, dishes shared with Fig. 7), which appear,
  from their titles and descriptions, to be in other dandisets.
- The array size, pixel size, packet, electrode positions and ROI measurements are not in the files.
- Subject ids are reused for different dishes, and the archive counts 19 subjects for 21 dishes.
- The dark floor of the images differs between dishes and sessions (33 to 81 a.u.), and the paper
  does not say whether a background was removed; the fold changes of fig7a depend on it.
- The 4.0 kV 1Px4 dishes, the Packet 2 condition of Fig. 8 if Fig. 8 used the small array (its
  legend does not say), read about 615 a.u. at the periphery
  and 719 a.u. in the centre, against about 120 and 215 to 235 a.u. in the paper.
- Of the 21 files, 7 name no detector, objective or channel.